In [6]:
# Data Loading

# import necessary libraries
from dotenv import load_dotenv
import os
from supabase import create_client
import pandas as pd

# Load environment variables and create Supabase client
load_dotenv()
supabase = create_client(os.getenv("SUPABASE_URL"), os.getenv("SUPABASE_KEY"))

# Function to fetch data from Supabase in chunks
def get_data(table_name):
    data = []
    page_size = 1000
    page = 0
    while True:
        response = supabase.table(table_name).select("*").range(page * page_size, (page + 1) * page_size - 1).execute()
        data.extend(response.data)
        if len(response.data) < page_size:
            break
        page += 1
    return pd.DataFrame(data)

# Fetch data from Supabase tables
df_sales = get_data("sales")
df_customers = get_data("customers")

# Merge the sales and customers dataframes on customer_id
df=pd.merge(df_sales, df_customers[['customer_id', 'first_name', 'last_name', 'email', 'phone', 'city']], on='customer_id', how='left')
#-------------

# Roll C
# 1. Viitekuupäev
today = pd.Timestamp('2025-02-28')
# Muuda sale_date tavaliseks pandas datetime tüübiks
df['sale_date'] = pd.to_datetime(
    df['sale_date'].astype(str),
    errors='coerce'
)

# 2. RECENCY
# Iga kliendi viimase ostu kuupäev
recency = (
    df.groupby('customer_id')['sale_date']
      .max()
      .reset_index()
)

recency['recency_days'] = (today - recency['sale_date']).dt.days

# Jätame alles ainult vajalikud veerud
recency = recency[['customer_id', 'recency_days']]


# 3. FREQUENCY
# Iga kliendi ostude arv
frequency = (
    df.groupby('customer_id')['sale_id']
      .count()
      .reset_index(name='frequency')
)


# 4. MONETARY
# Iga kliendi kogukulutus
monetary = (
    df.groupby('customer_id')['total_price']
      .sum()
      .reset_index(name='monetary')
)


# 5. Liidame R, F ja M üheks tabeliks
rfm = pd.merge(recency, frequency, on='customer_id')
rfm = pd.merge(rfm, monetary, on='customer_id')


# RFM skoorid 1–5

# Recency:
# mida väiksem päevade arv, seda parem ehk kõrgem skoor
rfm['R_score'] = pd.qcut(
    rfm['recency_days'],
    5,
    labels=[5, 4, 3, 2, 1]
)

# Frequency:
# suurem ostude arv = kõrgem skoor
# rank(method='first') aitab korduvate väärtuste korral
rfm['F_score'] = pd.qcut(
    rfm['frequency'].rank(method='first'),
    5,
    labels=[1, 2, 3, 4, 5]
)

# Monetary:
# suurem kogukulutus = kõrgem skoor
rfm['M_score'] = pd.qcut(
    rfm['monetary'],
    5,
    labels=[1, 2, 3, 4, 5]
)


# Muudame skoorid arvudeks
rfm['R_score'] = rfm['R_score'].astype(int)
rfm['F_score'] = rfm['F_score'].astype(int)
rfm['M_score'] = rfm['M_score'].astype(int)


# 6. RFM koguskoor
rfm['RFM_Score'] = (
    rfm['R_score']
    + rfm['F_score']
    + rfm['M_score']
)
# Segmenteerimisfunktsioon
def segment_customer(score):
    if score >= 13:
        return 'VIP Champions'
    elif score >= 10:
        return 'Loyal'
    elif score >= 7:
        return 'Potential'
    elif score >= 4:
        return 'At Risk'
    else:
        return 'Lost'


# Määra igale kliendile segment
rfm['Segment'] = rfm['RFM_Score'].apply(segment_customer)
# Segmentide kokkuvõte
segment_summary = (
    rfm.groupby('Segment')
       .agg(
           klientide_arv=('customer_id', 'count')
       )
       .reset_index()
)

# Osakaal protsentides
segment_summary['osakaal_pct'] = (
    segment_summary['klientide_arv']
    / segment_summary['klientide_arv'].sum()
    * 100
).round(2)

print(segment_summary)
# ============================================
# KVALITEEDIKONTROLL
# ============================================

print("Kliente kokku:", len(rfm))

print("\nR_score väärtused:")
print(rfm['R_score'].value_counts().sort_index())

print("\nF_score väärtused:")
print(rfm['F_score'].value_counts().sort_index())

print("\nM_score väärtused:")
print(rfm['M_score'].value_counts().sort_index())

print("\nPuuduvaid segmente:")
print(rfm['Segment'].isnull().sum())

print("\nSegmentide kokkuvõte:")
print(segment_summary)

print("\nRFM tabeli esimesed read:")
print(rfm.head())

rfm[['recency_days', 'frequency', 'monetary']].describe()

         Segment  klientide_arv  osakaal_pct
0        At Risk            535        20.97
1           Lost            109         4.27
2          Loyal            691        27.09
3      Potential            770        30.18
4  VIP Champions            446        17.48
Kliente kokku: 2551

R_score väärtused:
R_score
1    507
2    511
3    508
4    506
5    519
Name: count, dtype: int64

F_score väärtused:
F_score
1    511
2    510
3    510
4    510
5    510
Name: count, dtype: int64

M_score väärtused:
M_score
1    511
2    510
3    510
4    510
5    510
Name: count, dtype: int64

Puuduvaid segmente:
0

Segmentide kokkuvõte:
         Segment  klientide_arv  osakaal_pct
0        At Risk            535        20.97
1           Lost            109         4.27
2          Loyal            691        27.09
3      Potential            770        30.18
4  VIP Champions            446        17.48

RFM tabeli esimesed read:
   customer_id  recency_days  frequency  monetary  R_score  F_score  M

,recency_days,frequency,monetary
count,2551.000000,2551.000000,2551.000000
mean,195.941984,3.578989,1028.119083
std,174.834772,5.438059,1730.466498
min,-485.000000,1.000000,-1354.470000
25%,69.000000,2.000000,394.710000
50%,158.000000,3.000000,760.310000
75%,285.500000,4.000000,1275.270000
max,783.000000,78.000000,27668.020000
